# Tahap 1: Pengumpulan Data (Data Ingestion)
Pada tahap awal ini, kita mengumpulkan dan menyiapkan *dataset* mentah. Karena ini adalah simulasi, kita men-generate 100.000 baris data *dummy* yang merepresentasikan profil finansial nasabah, seperti pendapatan, total utang, jumlah pinjaman, tenor, bunga, dan status kredit. 

Data mentah ini kemudian langsung disimpan ke dalam format `.xlsx` (Excel) agar kita memiliki arsip *dataset* kotor sebelum dilakukan manipulasi apa pun.

In [6]:
import pandas as pd
import numpy as np

# 1. MENGUMPULKAN DATASET KOTOR
print("Memuat 100.000 baris data dummy...")
np.random.seed(42)
data_size = 100000 

df_raw = pd.DataFrame({
    'pendapatan_tahunan': np.random.randint(30000000, 200000000, data_size),
    'total_utang_saat_ini': np.random.randint(0, 50000000, data_size),
    'jumlah_pinjaman_diajukan': np.random.randint(5000000, 100000000, data_size),
    'tenor_bulan': np.random.choice([12, 24, 36, 48, 60], data_size),
    'bunga_tahunan': np.random.uniform(0.05, 0.20, data_size),
    'status_pekerjaan': np.random.choice(['Pegawai Tetap', 'Kontrak', 'Freelance', 'Wirausaha'], data_size),
    'status_kredit': np.random.choice(['Lancar', 'Macet'], data_size, p=[0.8, 0.2]) 
})

# Menyimpan dataset kotor 
nama_file_kotor = 'Dataset_Kotor_100k.xlsx'
print(f"Menyimpan ke {nama_file_kotor} (tunggu sekitar 1-2 menit)...")
df_raw.to_excel(nama_file_kotor, index=False)
print("Selesai! Silakan cek panel 'Output' untuk mendownloadnya.")

Memuat 100.000 baris data dummy...
Menyimpan ke Dataset_Kotor_100k.xlsx (tunggu sekitar 1-2 menit)...
Selesai! Silakan cek panel 'Output' untuk mendownloadnya.


# Tahap 2: Validasi Data Mentah (Exploratory Data Analysis / EDA)
Sebelum melakukan pembersihan, kita wajib mengetahui kondisi "kesehatan" data mentah kita. Langkah ini berfungsi sebagai radar untuk mendeteksi:
1. **Data Duplikat:** Baris data yang kembar dan berulang.
2. **Missing Values:** Sel data yang kosong.
3. **Anomali (Outlier):** Mengecek nilai minimum dan maksimum (misalnya, apakah ada nilai pinjaman yang tidak masuk akal atau pendapatan bernilai minus). Laporan ini akan menjadi acuan untuk tahap *cleaning*.

In [7]:
print("=== LAPORAN VALIDASI DATA MENTAH ===")

# 1. Pengecekan Data Duplikat
jumlah_duplikat = df_raw.duplicated().sum()
print(f"\n1. Jumlah Baris Duplikat: {jumlah_duplikat} baris")
if jumlah_duplikat > 0:
    print("   -> Terdapat data kembar. Data ini harus dihapus di tahap Cleaning.")
else:
    print("   -> Aman! Tidak ada data kembar.")

# 2. Pengecekan Data Kosong (Missing Values)
jumlah_kosong = df_raw.isnull().sum().sum()
print(f"\n2. Jumlah Data Kosong (Missing Values): {jumlah_kosong} sel")

# 3. Pengecekan Anomali (Outlier) dengan Deskripsi Statistik
print("\n3. Ringkasan Statistik untuk Deteksi Anomali:")
ringkasan = df_raw[['pendapatan_tahunan', 'total_utang_saat_ini', 'jumlah_pinjaman_diajukan']].describe().loc[['min', 'mean', 'max']]
print(ringkasan)

print("\nTips Membaca Anomali:")
print("- Cek baris 'min': Apakah ada pendapatan atau pinjaman yang bernilai negatif/minus atau Rp0?")
print("- Cek baris 'max': Apakah ada angka yang terlalu ekstrem (misal pinjaman terlalu tinggi)?")

=== LAPORAN VALIDASI DATA MENTAH ===

1. Jumlah Baris Duplikat: 0 baris
   -> Aman! Tidak ada data kembar.

2. Jumlah Data Kosong (Missing Values): 0 sel

3. Ringkasan Statistik untuk Deteksi Anomali:
      pendapatan_tahunan  total_utang_saat_ini  jumlah_pinjaman_diajukan
min         3.000058e+07          2.180000e+02              5.000826e+06
mean        1.149013e+08          2.502587e+07              5.245209e+07
max         1.999975e+08          4.999988e+07              9.999879e+07

Tips Membaca Anomali:
- Cek baris 'min': Apakah ada pendapatan atau pinjaman yang bernilai negatif/minus atau Rp0?
- Cek baris 'max': Apakah ada angka yang terlalu ekstrem (misal pinjaman terlalu tinggi)?


# Tahap 3: Data Cleaning & Feature Engineering
Berdasarkan laporan EDA sebelumnya, kita mengeksekusi pembersihan data dengan membuang duplikat dan menambal data kosong menggunakan nilai tengah (*median*). 

Setelah data bersih, kita masuk ke tahap **Feature Engineering** untuk mengkalkulasi 3 variabel finansial baru yang krusial untuk mengukur risiko kredit:
* **Monthly Installment:** Estimasi cicilan bulanan (pokok + bunga).
* **Debt-to-Income Ratio (DTI):** Rasio beban utang terhadap pendapatan bulanan.
* **Remaining Income:** Sisa uang nasabah setelah dipotong kewajiban utang.

In [8]:
# Membuat salinan data
df_clean = df_raw.copy()

print("Melakukan Data Cleaning...")
# Menghapus duplikat dan mengisi nilai kosong
df_clean.drop_duplicates(inplace=True)
df_clean.fillna(df_clean.median(numeric_only=True), inplace=True)

print("Melakukan Feature Engineering...")
# A. Monthly Installment (Cicilan Bulanan)
bunga_bulanan = df_clean['bunga_tahunan'] / 12
df_clean['monthly_installment'] = (df_clean['jumlah_pinjaman_diajukan'] / df_clean['tenor_bulan']) + (df_clean['jumlah_pinjaman_diajukan'] * bunga_bulanan)

# B. Debt-to-Income Ratio (DTI)
pendapatan_bulanan = df_clean['pendapatan_tahunan'] / 12
estimasi_cicilan_lama = df_clean['total_utang_saat_ini'] * 0.05 
df_clean['debt_to_income_ratio'] = (df_clean['monthly_installment'] + estimasi_cicilan_lama) / pendapatan_bulanan

# C. Remaining Income (Sisa Pendapatan)
df_clean['remaining_income'] = pendapatan_bulanan - (df_clean['monthly_installment'] + estimasi_cicilan_lama)

print("Data Cleaning & Feature Engineering Selesai!")

Melakukan Data Cleaning...
Melakukan Feature Engineering...
Data Cleaning & Feature Engineering Selesai!


# Tahap 4: Encoding & Transformasi (Standardisasi Skala)
Mesin tidak bisa membaca teks, dan mesin bisa menjadi bias jika membaca angka dengan rentang yang terlalu jauh berbeda (misal: membandingkan puluhan juta Rupiah dengan rasio 0.5).
Oleh karena itu, tahap ini melakukan dua hal:
1. **Label Encoding:** Mengubah kolom teks seperti status pekerjaan menjadi kode angka (0, 1, 2, dst.).
2. **Standard Scaler:** Mengubah skala variabel numerik menjadi *Z-score* (menghilangkan satuan Rupiah/persen dan meratakan titik tengah menjadi 0). Hal ini memastikan keadilan antar variabel saat dimasukkan ke dalam algoritma model.

In [9]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("Melakukan Data Encoding dan Transformasi...")

# Encoding teks ke angka
encoder = LabelEncoder()
df_clean['status_pekerjaan_encoded'] = encoder.fit_transform(df_clean['status_pekerjaan'])
df_clean['status_kredit_encoded'] = encoder.fit_transform(df_clean['status_kredit']) 

# Menghapus kolom teks yang sudah tidak terpakai
df_clean.drop(['status_pekerjaan', 'status_kredit'], axis=1, inplace=True)

# Normalisasi data numerik (Standardization)
scaler = StandardScaler()
kolom_numerik = ['pendapatan_tahunan', 'total_utang_saat_ini', 'jumlah_pinjaman_diajukan', 
                 'monthly_installment', 'debt_to_income_ratio', 'remaining_income']
df_clean[kolom_numerik] = scaler.fit_transform(df_clean[kolom_numerik])

print("\n=== Proses Prepocessing Selesai! Preview Data: ===")
print(df_clean.head())

Melakukan Data Encoding dan Transformasi...

=== Proses Prepocessing Selesai! Preview Data: ===
   pendapatan_tahunan  total_utang_saat_ini  jumlah_pinjaman_diajukan  \
0           -0.574009             -0.472575                 -1.695128   
1            0.869196              1.025339                 -0.730390   
2            1.435689              1.185078                 -1.437209   
3            1.277296              1.520983                  0.785214   
4            0.985485             -1.394249                  0.459548   

   tenor_bulan  bunga_tahunan  monthly_installment  debt_to_income_ratio  \
0           36       0.120334            -1.178834             -0.796572   
1           60       0.077144            -0.911686             -0.678668   
2           60       0.052428            -1.153710             -0.802538   
3           48       0.150874            -0.033059             -0.412676   
4           60       0.123818            -0.398235             -0.819079   

   remai

# Tahap 5: Ekspor Dataset Bersih (Siap ML)
 Data saat ini sudah 100% matang, bersih dari anomali, berbentuk numerik seutuhnya, dan diskalakan dengan baik. Langkah terakhir di *notebook* ini adalah mengekspor *dataframe* yang sudah rapi tersebut kembali ke dalam format `.xlsx`. File inilah yang nantinya akan di-*load* untuk melatih (*training*) model Machine Learning Random Forest.

In [10]:
# 5. MENYIMPAN DATASET BERSIH
nama_file_bersih = 'Dataset_Bersih_Siap_ML_100k.xlsx'

print(f"Menyimpan ke {nama_file_bersih} (tunggu sekitar 1-2 menit)...")
df_clean.to_excel(nama_file_bersih, index=False)
print(f"Selesai! Silakan cek panel 'Output' Kaggle di sebelah kanan untuk mendownload file data bersihnya.")
print("Data sudah matang dan siap dimasukkan ke model Machine Learning!")

Menyimpan ke Dataset_Bersih_Siap_ML_100k.xlsx (tunggu sekitar 1-2 menit)...
Selesai! Silakan cek panel 'Output' Kaggle di sebelah kanan untuk mendownload file data bersihnya.
Data sudah matang dan siap dimasukkan ke model Machine Learning!
